In [1]:
#import libraries
import pandas as pd
import requests
import random
import re
from transformers import pipeline
from newspaper import Article
import time
from datetime import datetime

from bs4 import BeautifulSoup

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import InvalidSessionIdException

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

import yfinance as yf
import urllib.parse

import matplotlib.pyplot as plt

import csv
import xml.etree.ElementTree as ET

c:\Users\badar\Desktop\VS Studio\Capstone 1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Search Keywords
keywords = ["Interest", "rate", "hike", "Overnight Policy Rate", "OPR", "ringgit", "depreciation", "GDP", "Budget", "fiscal", "deficit", "inflation", "spike", 
"foreign", "fund", "outflows","Bursa", "downtrend","Bank Negara", "bnm", "decision", "currency", "liquidity", "debt", "trade", "capitalization",
"Crude oil", "price drop", "palm oil", "export", "ban","Construction", "boost", "telecom", "downgrade","banking", "profit", "rise",
"GE14", "GE14", "elections", "political", "instability", "cabinet", "reshuffle","emergency", "declaration","policy", "uncertainty", "pandemic", "COVID"]

In [3]:
# Date range for filtering articles
START_DATE = datetime(2019, 1, 1)
END_DATE = datetime(2025, 1, 1)

# User-Agent header to mimic a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36"
}

# Base URL for Google News RSS search limited to Malaysia site
GOOG_RSS_URL = "https://news.google.com/rss/search?q={query}+site:.my&hl=en-MY&gl=MY&ceid=MY:en"

# Storage for results
all_articles = []

def parse_rss_date(date_str):
   
    try:
        return datetime.strptime(date_str, '%a, %d %b %Y %H:%M:%S %Z')
    except Exception:
        return None

def fetch_articles(keyword, max_retries=3):
    query = requests.utils.quote(keyword)
    url = GOOG_RSS_URL.format(query=query)
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
        except requests.RequestException as e:
            print(f"Request error for keyword '{keyword}': {e}")
            retries += 1
            time.sleep(random.uniform(2,5))
            continue

        if response.status_code == 200:
            try:
                root = ET.fromstring(response.content)
            except ET.ParseError:
                print(f"Failed to parse RSS for keyword '{keyword}'")
                return []
            articles_data = []
            for item in root.findall('.//item'):
                title_elem = item.find('title')
                link_elem = item.find('link')
                pubDate_elem = item.find('pubDate')

                if title_elem is None or link_elem is None or pubDate_elem is None:
                    continue
                title = title_elem.text
                link = link_elem.text
                pubDate_str = pubDate_elem.text
                pubDate = parse_rss_date(pubDate_str)
                if pubDate is None:
                    continue
                # Filter by date range
                if START_DATE <= pubDate <= END_DATE:
                    articles_data.append({
                        "keyword": keyword,
                        "headline": title,
                        "link": link,
                        "date": pubDate.strftime('%Y-%m-%d %H:%M:%S')
                    })
            print(f"Found {len(articles_data)} articles for keyword '{keyword}'")
            return articles_data
        elif response.status_code == 429:
            wait = random.uniform(5, 15)
            print(f"Rate limited (429) for {keyword}, retrying after {wait:.1f} seconds...")
            time.sleep(wait)
            retries += 1
        else:
            print(f"Failed to fetch page for {keyword} (status {response.status_code})")
            return []
    print(f"Exceeded max retries for keyword '{keyword}'")
    return []

for keyword in keywords:
    print(f"\nSearching articles for keyword: {keyword}")
    articles = fetch_articles(keyword)
    all_articles.extend(articles)
    time.sleep(random.uniform(1, 3))

print(f"\Total articles collected: {len(all_articles)}")

# Save to CSV
with open('news_headlines_1.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['keyword', 'headline', 'link', 'date']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in all_articles:
        writer.writerow(row)

print("Saved results to news_headlines_1.csv")


Searching articles for keyword: Interest
Found 2 articles for keyword 'Interest'

Searching articles for keyword: rate
Found 15 articles for keyword 'rate'

Searching articles for keyword: hike
Found 4 articles for keyword 'hike'

Searching articles for keyword: Overnight Policy Rate
Found 41 articles for keyword 'Overnight Policy Rate'

Searching articles for keyword: OPR
Found 73 articles for keyword 'OPR'

Searching articles for keyword: ringgit
Found 2 articles for keyword 'ringgit'

Searching articles for keyword: depreciation
Found 70 articles for keyword 'depreciation'

Searching articles for keyword: GDP
Found 2 articles for keyword 'GDP'

Searching articles for keyword: Budget
Found 47 articles for keyword 'Budget'

Searching articles for keyword: fiscal
Found 5 articles for keyword 'fiscal'

Searching articles for keyword: deficit
Found 38 articles for keyword 'deficit'

Searching articles for keyword: inflation
Found 2 articles for keyword 'inflation'

Searching articles fo